# Week 6 — Reinforcement Learning from First Principles

## Beginner-Friendly Theory, Practice, Q-Learning, Deep Q-Learning and a Real-World Transition Dataset

> **Teaching philosophy:** We will not rush into algorithms. First we understand the problem, the vocabulary, and the way an agent learns. Then we write small pieces of code. Only after the ideas are clear do we move to Q-Learning, neural networks, and a real-world-style dataset.

This notebook is designed to work in two ways:

- **Student workbook:** read the explanations, run the cells, answer the exercises, and complete the practicals.
- **Tutor guide:** each section contains teaching notes, common mistakes, discussion questions, and suggested explanations.

We will use a delivery/robot decision-making scenario throughout because it gives us a concrete story for abstract RL ideas.


## Learning Objectives

By the end of this notebook, a beginner should be able to:

1. Explain Reinforcement Learning in simple language.
2. Identify the **agent, environment, state, action, reward, episode, and policy** in a real problem.
3. Explain how RL differs from supervised and unsupervised learning.
4. Understand the **Markov Decision Process (MDP)** idea.
5. Explain exploration vs exploitation and epsilon-greedy action selection.
6. Understand rewards, returns, discounting, value functions, and Q-values.
7. Explain the Bellman idea without treating it as a formula to memorize.
8. Implement a small environment from scratch.
9. Implement and train **Q-Learning**.
10. Understand SARSA and how it differs from Q-Learning.
11. Understand why neural networks become useful when state spaces become large.
12. Build a simple **Deep Q-Network (DQN)** for an interactive environment.
13. Load, inspect, clean, encode, and validate an RL transition CSV.
14. Understand how a transition dataset can be used for an **offline/fitted Q-learning-style** experiment.
15. Recognize practical issues such as reward hacking, unsafe exploration, distribution shift, and the difference between a classroom experiment and production RL.


## How to Use This Notebook

### For students

Do not simply run every cell. For important sections, pause and answer the question before executing the code.

A useful learning rhythm is:

**Read → Predict → Code → Run → Interpret → Practice**

### For tutors

A good beginner lesson should repeatedly connect three layers:

1. **Story:** What is happening in the real world?
2. **RL language:** What are the state, action and reward?
3. **Code/math:** How do we represent that idea computationally?

If a learner is confused by a formula, return to the story. If the story is clear but the code is confusing, return to the state/action representation.


## Notebook Roadmap

| Part | Topic | Main learner output |
|---|---|---|
| 1 | RL intuition | Explain RL in your own words |
| 2 | RL vocabulary | Identify the seven core components |
| 3 | MDPs | Describe a sequential decision problem |
| 4 | Rewards and returns | Calculate simple returns |
| 5 | Policies and exploration | Implement epsilon-greedy selection |
| 6 | Value functions | Understand V(s) and Q(s,a) |
| 7 | Bellman thinking | Build a one-step target |
| 8 | GridWorld | Build an RL environment |
| 9 | Q-Learning | Train an agent from scratch |
| 10 | SARSA | Compare TD control methods |
| 11 | From tables to neural networks | Understand DQN motivation |
| 12 | DQN practical | Train a neural Q-network |
| 13 | Real-world CSV | Inspect and clean transitions |
| 14 | Offline/fitted Q-learning | Train from recorded transitions |
| 15 | Real-world engineering | Understand limitations and safety |
| 16 | Assessment | Exercises, project and reflection |


# Part 1 — What Is Reinforcement Learning?

## Start With a Familiar Story

Imagine you are teaching a child to ride a bicycle.

You do not give the child a table containing the correct answer for every possible situation. Instead, the child:

- observes what is happening,
- chooses an action,
- sees what happens next,
- receives feedback,
- and gradually changes future behaviour.

That is the basic idea behind Reinforcement Learning.


## A Simple Definition

**Reinforcement Learning (RL)** is a branch of Machine Learning in which an **agent learns how to make decisions by interacting with an environment and receiving feedback called rewards**.

The important words are:

- **agent** — the learner/decision-maker
- **environment** — the world the agent interacts with
- **state** — information describing the current situation
- **action** — a choice the agent can make
- **reward** — numerical feedback after an action
- **policy** — the agent's rule for choosing actions
- **episode** — one complete attempt or sequence of interaction

A useful mental model is:

**See → Choose → Act → Receive feedback → Learn → Repeat**


## What Makes RL Different?

In ordinary supervised learning, we commonly have:

**input → known correct answer**

For example:

`house features → house price`

In reinforcement learning, we usually have:

**current situation → action → consequence → reward**

The agent is not simply told the correct action for every situation. It has to learn which actions tend to produce better long-term outcomes.

### Important distinction

RL is therefore about **sequential decision-making**, not simply predicting a label.


## Three Learning Styles Side by Side

| Learning type | Main idea | Example |
|---|---|---|
| Supervised | Learn from labelled examples | Predict whether an email is spam |
| Unsupervised | Find structure without target labels | Group customers by behaviour |
| Reinforcement | Learn decisions from interaction/feedback | Learn how a robot should move |

**Tutor prompt:** Ask learners to give one example from their own lives for each learning style.


## Real-World RL Scenarios

RL can be studied for problems involving repeated decisions such as:

- robot navigation,
- game playing,
- traffic signal control,
- inventory decisions,
- resource allocation,
- scheduling,
- recommendation and ranking experiments,
- energy management,
- warehouse routing,
- autonomous systems.

Real-world systems add difficult constraints: safety, delayed outcomes, expensive experiments, changing environments, limited observations, and imperfect reward definitions.


## Warm-Up Question

Before running any code, answer:

> If an AI system controls a delivery robot, what could the **agent** be, and what could the **environment** be?

Write your answer in your own words.


In [ ]:
# Student answer area
agent = ""
environment = ""

print("Agent:", agent)
print("Environment:", environment)

## Understanding the Code

This is a student practice cell for identifying the **agent** and **environment**.

- The variables provide places to enter the learner and the world it interacts with.
- `print()` displays the answers.

### Observation

Replace the empty values with a real-world RL example and explain why one is the agent and the other is the environment.


### Tutor answer guide

A reasonable answer is:

- **Agent:** the delivery robot's decision-making system.
- **Environment:** the warehouse/city/road network and the changing conditions around the robot.

There can be different valid designs. For example, the agent could be the whole robot controller, or a particular decision-making module inside it.


# Part 2 — The Seven Core RL Words

We will use one scenario repeatedly: **a delivery robot deciding what to do next.**

Suppose the robot currently has 25% battery, 8 pending orders, moderate traffic, and 5 km to its next target.


## 1. Agent

The **agent** is the learner or decision-maker.

In our example:

> The delivery robot's decision-making program is the agent.

The agent observes information and chooses actions.


## 2. Environment

The **environment** is everything the agent interacts with and everything that determines what happens after an action.

For a delivery robot, this might include:

- roads or warehouse paths,
- traffic,
- weather,
- battery effects,
- orders,
- customers,
- other robots.


## 3. State

A **state** is a representation of the situation at a particular moment.

For example:

```text
battery = 25%
pending_orders = 8
traffic = 0.57
distance = 5.0 km
hour = 14:00
```

The state should contain information useful for deciding what to do next.

### Beginner analogy

Think of a state as a **snapshot of the world right now**.


## 4. Action

An **action** is a choice available to the agent.

For our robot, actions might be:

- `deliver`
- `recharge`
- `reroute`
- `wait`

The set of possible actions is called the **action space**.


## 5. Reward

A **reward** is numerical feedback from the environment.

For example:

- successful delivery: `+5`
- unnecessary delay: `-1`
- unsafe decision: `-10`

The reward is not necessarily the same thing as money. It is a numerical signal that represents what the learning system is being encouraged or discouraged to do.

### Critical idea

The agent does not automatically understand your real-world goal. It learns from the reward signal you give it.


## 6. Policy

A **policy** is the agent's strategy for choosing an action when it sees a state.

In simple language:

> **Policy = what should I do when I see this situation?**

A policy might eventually learn:

```text
low battery + far from charger → recharge
high demand + enough battery → deliver
heavy traffic + distant destination → reroute
```


## 7. Episode

An **episode** is one complete run of interaction.

Examples:

- one game from start to finish,
- one robot delivery mission,
- one simulation run,
- one customer-session decision sequence.

An episode usually ends when a terminal condition occurs.


## The Full RL Loop

```text
              ┌───────────────┐
              │  Environment  │
              └───────┬───────┘
                      │ state
                      ▼
                 ┌─────────┐
                 │  Agent  │
                 └────┬────┘
                      │ action
                      ▼
              ┌───────────────┐
              │  Environment  │
              └───────┬───────┘
                      │
                reward + next state
                      │
                      └──────────► Agent learns
```

The cycle repeats many times.


## Check Your Understanding

For a traffic-light controller:

1. What is the agent?
2. What is the environment?
3. Give two possible state variables.
4. Give three possible actions.
5. Give one positive and one negative reward.
6. What could count as an episode?

**Do not continue until you can answer these without looking at the previous section.**


# Part 3 — Markov Decision Processes (MDPs)

A large amount of classical RL is described using a **Markov Decision Process**, usually shortened to **MDP**.

Do not let the name intimidate you.

An MDP is simply a mathematical way of describing a sequential decision problem.


## The Markov Idea

The key intuition is:

> The current state should contain the information needed to predict what matters about the future.

Imagine a robot state contains:

- current location,
- battery level,
- traffic,
- pending orders.

If those variables contain enough relevant information, the agent does not need the entire history of every previous movement to make its next decision.


## Why State Design Matters

A poor state representation can make learning difficult.

Suppose a robot must decide whether to recharge, but the state contains no battery information.

The model cannot reliably learn a rule involving battery level because it has not been given that information.

**Good RL engineering starts with a useful representation of the decision situation.**


## MDP Components

A beginner-friendly description is:

- **S** — possible states
- **A** — possible actions
- **R** — rewards
- **P** — transition dynamics: what can happen after an action
- **γ (gamma)** — discount factor for future rewards

You do not need to memorize these symbols yet. Focus on the story first.


## State vs Observation

These words are related but not always identical.

- **State:** the underlying situation relevant to the decision process.
- **Observation:** the information the agent actually receives.

If a robot has a hidden sensor fault, the full state might include that fault while the agent's observation does not.

When observations are incomplete, we enter the area of **partial observability**, which makes RL more difficult.


# Part 4 — Rewards, Returns and Discounting

## Immediate Reward vs Long-Term Outcome

Suppose a robot has two choices:

**Action A:** earn `+5` now but create a bad future situation.

**Action B:** earn `-1` now by recharging but become able to earn many rewards later.

RL therefore needs a way to reason about **future rewards**, not only the reward received immediately.


## Return

The **return** is the total reward an agent receives from a point in time onward, usually with discounting.

A simplified version is:

`G = rₜ + γrₜ₊₁ + γ²rₜ₊₂ + ...`

Read this as:

> reward now + discounted reward later + more heavily discounted reward after that.


## What Is Gamma (γ)?

The discount factor `gamma` is usually between 0 and 1.

- `gamma = 0` means the agent cares only about immediate reward.
- a value close to `1` gives more importance to future rewards.

### Analogy

Imagine choosing between:

- receiving ₦1,000 today, or
- receiving ₦1,000 much later.

The importance you place on the later reward depends on your decision problem. Gamma gives an RL system a mathematical way to express that preference.


In [ ]:
gamma = 0.9
rewards = [5, 2, 3]

G = rewards[0] + gamma * rewards[1] + (gamma ** 2) * rewards[2]
print("Discounted return:", round(G, 3))

## Understanding the Code

This cell calculates **discounted return**.

- `gamma` is the discount factor.
- The immediate reward keeps its full value.
- Later rewards are multiplied by higher powers of `gamma`, so their contribution is discounted.

The pattern is `G = Rₜ + γRₜ₊₁ + γ²Rₜ₊₂ + ...`.

### Observation

Change `gamma` and observe how much the future rewards contribute to the final return.


## Exercise — Calculate a Return

Suppose:

- reward now = `4`
- next reward = `3`
- third reward = `2`
- `gamma = 0.5`

Calculate the return by hand before running the code.


In [ ]:
gamma = 0.5
rewards = [4, 3, 2]

answer = rewards[0] + gamma * rewards[1] + (gamma ** 2) * rewards[2]
print(answer)

## Understanding the Code

This cell calculates **discounted return**.

- `gamma` is the discount factor.
- The immediate reward keeps its full value.
- Later rewards are multiplied by higher powers of `gamma`, so their contribution is discounted.

The pattern is `G = Rₜ + γRₜ₊₁ + γ²Rₜ₊₂ + ...`.

### Observation

Change `gamma` and observe how much the future rewards contribute to the final return.


# Part 5 — Policies and Exploration

## What Is a Policy?

A policy tells the agent how to choose an action from a state.

We can write this conceptually as:

`policy(state) → action`

For a learned policy, the action may be selected from estimated values rather than from a hand-written rule.


## Exploration vs Exploitation

This is one of the most important ideas in RL.

### Exploitation

> Choose what currently appears to be the best action.

### Exploration

> Try an action we are less certain about so that we can learn more.

### Everyday analogy

Imagine a student choosing a restaurant.

- Exploitation: go to the restaurant they already know they like.
- Exploration: try a new restaurant to discover whether it is better.

RL needs a balance between the two.


## Epsilon-Greedy

A common beginner-friendly exploration method is **epsilon-greedy**.

If epsilon is high:

> explore more often.

If epsilon is low:

> exploit the best-known action more often.

For example, with `epsilon = 0.2`, the agent explores about 20% of the time in the simple textbook interpretation.


In [ ]:
import random

q_values = [1.2, 3.8, 2.1, 0.5]
epsilon = 0.2

def epsilon_greedy(q_values, epsilon):
    if random.random() < epsilon:
        return random.randrange(len(q_values)), "explore"
    return max(range(len(q_values)), key=lambda i: q_values[i]), "exploit"

for _ in range(10):
    action, mode = epsilon_greedy(q_values, epsilon)
    print("action:", action, "mode:", mode)

## Understanding the Code

This cell demonstrates **epsilon-greedy action selection**, which balances exploration and exploitation.

- `q_values` stores the agent's current estimated value for each action.
- `epsilon = 0.2` means the agent explores randomly about 20% of the time.
- `random.random()` generates a random number used to decide whether to explore.
- `random.randrange(...)` chooses a random action when exploring.
- `max(..., key=...)` chooses the action with the highest Q-value when exploiting.
- The loop repeats the decision so we can observe both behaviours.

### Observation

Run the cell several times. Because exploration is random, the sequence can change from run to run. Look for both `explore` and `exploit` decisions.


## A Beginner Mistake

Do not say:

> "Exploration means random behaviour forever."

A learning agent normally **reduces unnecessary exploration over time** or uses a strategy appropriate to the problem.

Exploration is useful because the agent does not know everything at the beginning.


# Part 6 — Value Functions and Q-Values

Now we need a way to answer:

> **How good is this situation?**

and later:

> **How good is this action in this situation?**


## State-Value Function V(s)

The **state-value function** estimates how good it is to be in a state.

`V(s)` means approximately:

> expected future return starting from state `s`, under a particular policy.

Example:

A robot at 90% battery near many orders may be in a high-value state.


## Action-Value Function Q(s, a)

The **Q-value** estimates how good it is to take action `a` while in state `s`.

`Q(s, a)` means:

> expected future return if I am in state `s`, take action `a`, and continue appropriately afterward.

### Beginner shortcut

- `V(s)` → how good is this **situation**?
- `Q(s,a)` → how good is this **choice in this situation**?


## Why Q-Values Are Useful

Suppose the robot is in state `S1`.

| Action | Q-value |
|---|---:|
| deliver | 4.2 |
| recharge | 7.8 |
| reroute | 5.1 |
| wait | 1.0 |

The current best-known action is `recharge` because its Q-value is highest.

But remember: these numbers are **estimates**, not magic truths. The learning algorithm has to improve them from experience.


In [ ]:
q_values = {
    "deliver": 4.2,
    "recharge": 7.8,
    "reroute": 5.1,
    "wait": 1.0
}

best_action = max(q_values, key=q_values.get)
print("Best known action:", best_action)
print("Estimated value:", q_values[best_action])

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


# Part 7 — Bellman Thinking

The Bellman idea is central to many RL algorithms.

Instead of trying to calculate an entire future story every time, we can think recursively:

> **How good is this action now, plus how valuable is the next situation?**

For Q-Learning, the core target is:

`target = reward + gamma × best estimated next Q-value`

for a non-terminal transition.


## One-Step Example

Suppose:

- current reward = `4`
- gamma = `0.9`
- best next Q-value = `6`

Then:

`target = 4 + 0.9 × 6 = 9.4`

The algorithm uses `9.4` as a target for improving the current Q estimate.


In [ ]:
reward = 4
gamma = 0.9
best_next_q = 6

target = reward + gamma * best_next_q
print("Bellman target:", target)

## Understanding the Code

This cell calculates a **Bellman target**.

- `reward` is the immediate feedback.
- `gamma` controls the importance of future value.
- `best_next_q` is the largest estimated Q-value in the next state.
- The target combines immediate reward and discounted future value.

### Observation

Change `gamma` or `best_next_q` and observe how the learning target changes.


## The TD Error

If the current Q-value is `7` and our new target is `9.4`, then the difference is:

`TD error = 9.4 - 7 = 2.4`

The algorithm can use that error to adjust its estimate.

This is the basic learning idea behind **Temporal-Difference (TD) learning**.


In [ ]:
current_q = 7
target = 9.4

td_error = target - current_q
print("TD error:", td_error)

## Understanding the Code

The **TD error** measures how far the current estimate is from the new target.

- `current_q` is the old estimate.
- `target` is the new Bellman target.
- `target - current_q` gives the correction signal.

A positive error suggests increasing the estimate; a negative error suggests decreasing it.

### Observation

Change `current_q` and observe how both the size and direction of the TD error change.


# Part 8 — Temporal-Difference Learning

## Monte Carlo vs Temporal Difference

Two useful learning styles are:

### Monte Carlo (MC)
Wait until the episode finishes and use the observed outcome.

### Temporal Difference (TD)
Update after a step using a mixture of immediate reward and an estimate of future value.

### Mental model

- **MC:** “Let me see how the whole story ends.”
- **TD:** “I can learn something from what just happened, even before the whole story ends.”


## SARSA vs Q-Learning

Both are TD control algorithms.

### SARSA
Uses the action the current policy actually selects next.

`Q(s,a) ← Q(s,a) + α[r + γQ(s',a') − Q(s,a)]`

### Q-Learning
Uses the best estimated next action.

`Q(s,a) ← Q(s,a) + α[r + γ max Q(s',a') − Q(s,a)]`

A standard teaching distinction is:

- SARSA → **on-policy**
- Q-Learning → **off-policy**


## What Does α (Alpha) Mean?

`alpha` is the learning rate.

It controls how strongly a new learning signal changes the current estimate.

- Small alpha → cautious updates.
- Large alpha → faster, larger updates.

This is another parameter that must be chosen thoughtfully.


# Part 9 — Build a Tiny Environment Before Building a Big Model

A common beginner mistake is to start with a huge dataset or neural network.

Instead, we will create a tiny **GridWorld** where every decision is visible.

Imagine a robot on a 5 × 5 grid.

- Start at the bottom-left.
- Goal at the top-right.
- Some cells are blocked.
- Reaching the goal gives a positive reward.
- Every normal step has a small cost.

The agent must discover a useful route.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class GridWorld:
    def __init__(self, rows=5, cols=5, start=(4, 0), goal=(0, 4),
                 obstacles={(2, 1), (2, 2), (2, 3)}):
        self.rows = rows
        self.cols = cols
        self.start = start
        self.goal = goal
        self.obstacles = set(obstacles)
        self.n_actions = 4
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        moves = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
        dr, dc = moves[action]
        candidate = (self.state[0] + dr, self.state[1] + dc)

        invalid = (
            candidate[0] < 0 or candidate[0] >= self.rows or
            candidate[1] < 0 or candidate[1] >= self.cols or
            candidate in self.obstacles
        )
        if invalid:
            candidate = self.state

        self.state = candidate
        if self.state == self.goal:
            return self.state, 10.0, True
        return self.state, -0.1, False

    def render(self):
        grid = np.zeros((self.rows, self.cols))
        for r, c in self.obstacles:
            grid[r, c] = -1
        sr, sc = self.start
        gr, gc = self.goal
        grid[sr, sc] = 1
        grid[gr, gc] = 2
        plt.figure(figsize=(6, 6))
        plt.imshow(grid)
        plt.xticks(range(self.cols))
        plt.yticks(range(self.rows))
        plt.grid(True)
        plt.title("GridWorld: Start=1, Goal=2, Obstacles=-1")
        plt.show()


env = GridWorld()
env.render()

## Understanding the Code

This cell creates a small **GridWorld environment** for practising reinforcement learning.

- `__init__()` defines the grid size, starting point, goal, obstacles, and available actions.
- `self.state` stores the agent's current position.
- `reset()` starts a new episode from the beginning.
- `step(action)` applies an action and returns the next state, reward, and `done` flag.

### Why this matters

An RL environment must respond to actions with a new state and feedback. These rules define the learning problem.

### Observation

Read the `step()` method and identify what happens when the agent moves normally, hits an obstacle, or reaches the goal.


## Understanding the Environment

For this environment:

- **state** = `(row, column)`
- **actions** = up, down, left, right
- **reward** = `10` at the goal and `-0.1` for a normal step
- **terminal state** = reaching the goal

This small environment is valuable because we can understand every part of it before introducing a neural network.


## Exercise — Step Through the Environment

Try actions manually.

Action IDs:

- `0` = up
- `1` = down
- `2` = left
- `3` = right

Predict the next state before running the code.


In [ ]:
env = GridWorld()
state = env.reset()
print("Starting state:", state)

for action in [0, 0, 0, 3]:
    next_state, reward, done = env.step(action)
    print(f"action={action}, next_state={next_state}, reward={reward}, done={done}")

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


# Part 10 — Q-Learning From Scratch

Now that we understand the environment, we can build the learning algorithm.

## The Q-Table

Because our GridWorld is tiny, we can store Q-values in a table.

Think of the table as a notebook containing questions like:

> “If I am at this position and I move right, how promising is that choice?”

Initially, the agent knows very little, so Q-values start at zero.


In [ ]:
from collections import defaultdict

q_table = defaultdict(lambda: np.zeros(4))

print("Q-values for the starting state:", q_table[(4, 0)])

## Understanding the Code

This cell creates the Q-table used by tabular Q-Learning.

- Each state gets an array of Q-values.
- `np.zeros(4)` creates one value for each of the four possible actions.
- Initially all values are zero because the agent has not learned yet.

### Observation

Think of the Q-table as the agent's memory: `state → value for each possible action`.


## Choosing an Action

We combine the Q-table with epsilon-greedy exploration.

At the beginning, we explore more. As learning progresses, we gradually reduce epsilon so the agent increasingly uses what it has learned.


In [ ]:
import random

def choose_action(q_table, state, epsilon, n_actions=4):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return int(np.argmax(q_table[state]))

print("Example action:", choose_action(q_table, (4, 0), epsilon=1.0))

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


## The Q-Learning Update

The update is:

`Q(s,a) ← Q(s,a) + α[target − Q(s,a)]`

where:

`target = r + γ max Q(s',a')`

for a non-terminal next state.

In plain English:

> Move the old estimate a little closer to the new target.


In [ ]:
def q_learning_update(q_table, state, action, reward, next_state,
                      alpha=0.1, gamma=0.95, done=False):
    best_next = 0 if done else np.max(q_table[next_state])
    target = reward + gamma * best_next
    td_error = target - q_table[state][action]
    q_table[state][action] += alpha * td_error
    return td_error

q = defaultdict(lambda: np.zeros(4))
td = q_learning_update(q, (4,0), 0, -0.1, (3,0))
print("TD error:", td)
print("Updated Q-values:", q[(4,0)])

## Understanding the Code

This function performs one **Q-Learning update**.

- The current Q-value represents what the agent currently believes.
- `best_next` finds the best estimated future value.
- `target = reward + gamma * best_next` applies the Bellman idea.
- `td_error` measures the difference between target and current estimate.
- `alpha` controls how large the update is.

### Observation

Run the cell and compare the Q-value before and after the update. One update is a tiny learning step; training repeats this process many times.


## Train the Agent

We now repeat the loop many times:

1. Reset the environment.
2. Observe the state.
3. Choose an action.
4. Take the action.
5. Receive reward and next state.
6. Update the Q-table.
7. Continue until the episode ends.


In [ ]:
def train_q_learning(env, episodes=1000, alpha=0.1, gamma=0.95,
                     epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995,
                     max_steps=100):
    q = defaultdict(lambda: np.zeros(env.n_actions))
    rewards_per_episode = []
    steps_per_episode = []

    for episode in range(episodes):
        state = env.reset()
        total_reward = 0

        for step in range(max_steps):
            action = choose_action(q, state, epsilon, env.n_actions)
            next_state, reward, done = env.step(action)
            q_learning_update(q, state, action, reward, next_state,
                              alpha=alpha, gamma=gamma, done=done)
            state = next_state
            total_reward += reward
            if done:
                break

        rewards_per_episode.append(total_reward)
        steps_per_episode.append(step + 1)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return q, rewards_per_episode, steps_per_episode

q_table, episode_rewards, episode_steps = train_q_learning(env)
print("Training complete.")

## Understanding the Code

This function performs one **Q-Learning update**.

- The current Q-value represents what the agent currently believes.
- `best_next` finds the best estimated future value.
- `target = reward + gamma * best_next` applies the Bellman idea.
- `td_error` measures the difference between target and current estimate.
- `alpha` controls how large the update is.

### Observation

Run the cell and compare the Q-value before and after the update. One update is a tiny learning step; training repeats this process many times.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.35, label="Episode reward")
window = 50
if len(episode_rewards) >= window:
    moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode="valid")
    plt.plot(range(window-1, len(episode_rewards)), moving_avg, label="50-episode average")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Q-Learning Training Progress")
plt.legend()
plt.show()

## Understanding the Code

This cell visualises learning progress.

- The raw episode rewards show how much reward the agent received in each episode.
- A moving average smooths the noisy RL signal.
- Axis labels and the title make the graph easier to interpret.

### Observation

Look at the overall trend rather than expecting every episode to improve. RL training is naturally noisy.


## What Should We Look For?

We are not looking for a perfect straight line.

We want evidence that the agent is increasingly discovering useful behaviour.

Possible signs include:

- higher rewards,
- fewer steps to the goal,
- more frequent successful episodes,
- a sensible policy in the grid.

A noisy learning curve is normal.


## Extract the Learned Policy

For each state, choose the action with the highest Q-value.

This turns the Q-table into a simple policy.


In [ ]:
arrows = {0: "↑", 1: "↓", 2: "←", 3: "→"}

policy = np.full((env.rows, env.cols), " ", dtype=object)
for r in range(env.rows):
    for col in range(env.cols):
        state = (r, col)
        if state in env.obstacles:
            policy[r, col] = "█"
        elif state == env.goal:
            policy[r, col] = "G"
        else:
            policy[r, col] = arrows[int(np.argmax(q_table[state]))]

print(policy)

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


## Practical Check

Starting from `(4, 0)`, follow the learned arrows manually.

Ask:

- Does the policy move toward the goal?
- Does it avoid obstacles?
- Does it appear to prefer shorter routes?
- What happens if you train for too few episodes?

This is an important RL skill: **inspect the behaviour, not only the metric.**


# Part 11 — Why Do We Need Deep Reinforcement Learning?

Our Q-table worked because GridWorld is tiny.

But imagine the state contains:

- battery level,
- GPS coordinates,
- traffic score,
- weather,
- demand,
- distance,
- time of day,
- hundreds of sensor readings.

A table can become enormous or impossible to maintain.

Instead of storing one number for every state-action pair, we can use a **neural network to approximate Q-values**.


## What Is a DQN?

**DQN = Deep Q-Network.**

The basic idea is:

> Use a neural network to estimate Q-values instead of storing them in a giant table.

Input:

`state`

Output:

`Q(state, action_1), Q(state, action_2), ...`

The action with the largest predicted Q-value is the current best-known choice.


## DQN Components

A beginner-friendly DQN implementation commonly introduces:

1. **Q-network** — predicts Q-values.
2. **Experience replay** — stores previous transitions and samples them for training.
3. **Target network** — a more stable copy used to calculate learning targets.
4. **Epsilon-greedy exploration** — balances exploration and exploitation.
5. **Bellman target** — connects reward and future value.

These are not separate algorithms floating in isolation. They are engineering techniques that help neural-network Q-learning train more stably.


## Experience Replay Analogy

Imagine studying from your last 10,000 practice questions instead of only studying the question you answered five seconds ago.

A replay buffer stores transitions such as:

`(state, action, reward, next_state, done)`

The learner randomly samples a batch of old experiences.

This helps break up strong correlations between consecutive experiences.


## Target Network Analogy

Imagine trying to hit a moving target while the target changes every time you take a step.

Training can become unstable.

A target network provides a more slowly changing reference for calculating targets.

In a standard educational DQN implementation, the target network is periodically updated from the online network.


# Part 12 — DQN Practical With GridWorld

We will keep the environment small so that students can understand the learning loop.

The neural network will receive a one-hot representation of the grid position and output four Q-values.


## Why We Use Keras in This Notebook

This course uses **Keras with TensorFlow** for the neural-network part of Reinforcement Learning. Keras lets us build neural networks in a clear, beginner-friendly way while keeping the underlying RL ideas rigorous.

The key idea is: **RL decides what the learning problem is; Keras helps us build the neural network that estimates Q-values.**


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from collections import deque
import random

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)


## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


In [ ]:
def build_q_network(state_size, action_size):
    """Build a neural network that predicts one Q-value per action."""
    model = keras.Sequential([
        layers.Input(shape=(state_size,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(action_size, activation="linear")
    ])
    return model

def state_to_vector(state, rows=5, cols=5):
    vector = np.zeros(rows * cols, dtype=np.float32)
    vector[state[0] * cols + state[1]] = 1.0
    return vector

network = build_q_network(25, 4)
network.summary()


## Understanding the Code

This cell builds the neural network used by DQN to estimate Q-values.

- `keras.Sequential(...)` stacks the layers in order.
- `layers.Input(...)` defines the number of input features.
- The two `Dense(64, activation="relu")` layers learn non-linear patterns.
- The final `Dense(action_size, activation="linear")` layer outputs one Q-value per available action.
- `state_to_vector()` converts a grid coordinate into a numerical representation the network can process.

### Key connection

`state → neural network → Q-value for each action`

### Observation

Run `network.summary()` and identify the input, hidden, and output layers.


## Why One-Hot Encode the Grid Position?

The neural network expects numbers.

A state such as `(4, 0)` is a coordinate, but we can represent it as a vector with one `1` and the rest `0`.

This is called **one-hot encoding**.

For this tiny teaching environment it is simple and transparent. Real systems may use continuous numerical features, embeddings, images, recurrent networks, or other representations.


In [ ]:
state_vector = state_to_vector((4, 0))
print(state_vector)
print("Vector length:", len(state_vector))

## Understanding the Code

This cell tests the one-hot state representation.

- `state_to_vector(...)` converts a grid position into numbers.
- `print(state_vector)` lets us inspect those numbers.
- `len(state_vector)` confirms the vector length matches the number of grid cells.

### Observation

You should see one `1` and many `0`s. Change the state and identify which position becomes `1`.


## Replay Buffer

A replay buffer is simply a collection of past experiences.

We will store:

`state, action, reward, next_state, done`

Then sample a random batch for learning.


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

buffer = ReplayBuffer()
print("Empty buffer size:", len(buffer))

## Understanding the Code

This class implements **experience replay**.

- `deque(maxlen=capacity)` keeps a fixed-size collection of experiences.
- `add()` stores `(state, action, reward, next_state, done)`.
- `sample()` randomly selects a batch for training.
- `__len__()` reports how many experiences are currently stored.

### Why random replay?

Consecutive RL experiences can be strongly related. Random sampling gives the neural network a more mixed training batch.

### Observation

The empty buffer should initially have size `0`. Later, watch the buffer grow as the agent interacts with the environment.


## DQN Training Target

For a non-terminal transition:

`target = reward + gamma × max Q_target(next_state, next_action)`

For a terminal transition:

`target = reward`

The online network predicts the Q-value for the action that was actually taken. The loss measures how far that prediction is from the target.


In [ ]:
def train_dqn(episodes=500, batch_size=64, gamma=0.95,
              lr=1e-3, epsilon=1.0, epsilon_min=0.05,
              epsilon_decay=0.995, target_update=20):
    """Train a small DQN using Keras and TensorFlow GradientTape."""
    env = GridWorld()
    state_size = env.rows * env.cols
    action_size = env.n_actions

    online = build_q_network(state_size, action_size)
    target = build_q_network(state_size, action_size)
    target.set_weights(online.get_weights())

    optimizer = keras.optimizers.Adam(learning_rate=lr)
    loss_fn = keras.losses.MeanSquaredError()
    replay = ReplayBuffer()
    history = []

    for episode in range(episodes):
        state = env.reset()
        total_reward = 0.0

        for _ in range(100):
            # Epsilon-greedy action selection.
            if random.random() < epsilon:
                action = random.randrange(action_size)
            else:
                state_vector = state_to_vector(state)[None, :]
                q_values = online(state_vector, training=False).numpy()[0]
                action = int(np.argmax(q_values))

            next_state, reward, done = env.step(action)
            replay.add(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward

            if len(replay) >= batch_size:
                states, actions, rewards, next_states, dones = replay.sample(batch_size)
                states_t = np.array([state_to_vector(s) for s in states], dtype=np.float32)
                next_states_t = np.array([state_to_vector(s) for s in next_states], dtype=np.float32)
                actions_t = np.array(actions, dtype=np.int32)
                rewards_t = np.array(rewards, dtype=np.float32)
                dones_t = np.array(dones, dtype=np.float32)

                # Compute the Bellman targets with the target network.
                next_q = target(next_states_t, training=False).numpy()
                max_next_q = np.max(next_q, axis=1)
                targets = rewards_t + gamma * max_next_q * (1.0 - dones_t)

                # Update only the Q-value of the action actually taken.
                with tf.GradientTape() as tape:
                    all_q = online(states_t, training=True)
                    action_mask = tf.one_hot(actions_t, depth=action_size)
                    current_q = tf.reduce_sum(all_q * action_mask, axis=1)
                    loss = loss_fn(targets, current_q)

                gradients = tape.gradient(loss, online.trainable_variables)
                optimizer.apply_gradients(zip(gradients, online.trainable_variables))

            if done:
                break

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        if episode % target_update == 0:
            target.set_weights(online.get_weights())

        history.append(total_reward)

    return online, history

online_net, dqn_rewards = train_dqn()
print("DQN training complete.")


## Understanding the Code

This is the main **Keras/TensorFlow DQN training loop**.

- The `online` network is trained to estimate Q-values.
- The `target` network provides a more stable source for Bellman targets.
- `Adam` updates the neural-network weights.
- `ReplayBuffer` stores previous transitions and supplies random training batches.
- Epsilon-greedy behaviour balances exploration and exploitation.
- `GradientTape` calculates gradients of the loss with respect to the online network's weights.
- The target network is periodically updated from the online network.

### The complete DQN cycle

`interact → store → sample → calculate target → calculate loss → update online network → update target network`

### Observation

Trace one training episode and identify where the state, action, reward, next state, Bellman target, loss, and weight update occur.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(dqn_rewards, alpha=0.35)
if len(dqn_rewards) >= 50:
    avg = np.convolve(dqn_rewards, np.ones(50)/50, mode="valid")
    plt.plot(range(49, len(dqn_rewards)), avg)
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("DQN Training Reward")
plt.show()

## Understanding the Code

This cell visualises learning progress.

- The raw episode rewards show how much reward the agent received in each episode.
- A moving average smooths the noisy RL signal.
- Axis labels and the title make the graph easier to interpret.

### Observation

Look at the overall trend rather than expecting every episode to improve. RL training is naturally noisy.


## DQN Evaluation

During evaluation, we normally reduce or remove exploration.

The goal is to measure the learned policy rather than the agent's ability to randomly explore.


In [ ]:
def evaluate_dqn(model, episodes=50):
    env = GridWorld()
    successes = 0
    rewards = []
    steps = []

    for _ in range(episodes):
        state = env.reset()
        total = 0

        for step in range(100):
            # No random exploration during evaluation.
            state_vector = state_to_vector(state)[None, :]
            q_values = model(state_vector, training=False).numpy()[0]
            action = int(np.argmax(q_values))

            state, reward, done = env.step(action)
            total += reward

            if done:
                successes += 1
                steps.append(step + 1)
                break

        rewards.append(total)

    return {
        "success_rate": successes / episodes,
        "mean_reward": np.mean(rewards),
        "mean_steps_when_successful": np.mean(steps) if steps else np.nan
    }

evaluation = evaluate_dqn(online_net)
evaluation


## Understanding the Code

This function evaluates a trained DQN with intentional exploration removed.

- The model predicts Q-values for the current state.
- `np.argmax(...)` selects the highest-valued action.
- `successes` counts completed goal-reaching episodes.
- `rewards` records total rewards.
- `steps` records how long successful episodes take.

### Observation

Evaluation measures the behaviour of the learned policy rather than mixing it with random exploration.


### Tutor discussion

Ask students why the evaluation function does not use the same random exploration rate as training.

Expected idea:

> During evaluation we want to measure what the learned policy does, not mix its performance with unnecessary random exploration.


# Part 13 — Our Real-World-Style RL Dataset

We now move from a tiny simulated environment to a transition dataset that looks more like an operational log.

The file is:

`delivery_rl_transition_log.csv`

Each row represents a transition:

**current state + action → reward + next state**

This is an important shift in perspective.


## What Is a Transition?

A transition is one step of experience.

We can represent it as:

`(s, a, r, s', done)`

where:

- `s` = current state
- `a` = action
- `r` = reward
- `s'` = next state
- `done` = whether the episode ended

This five-part structure is one of the most important patterns to recognize in RL data.


## Dataset Columns

The dataset contains fields such as:

- `episode_id`
- `timestamp`
- `warehouse_id`
- `robot_id`
- `zone`
- `weather_condition`
- `hour_of_day`
- `battery_pct`
- `demand_score`
- `traffic_score`
- `distance_to_goal_km`
- `pending_orders`
- `action`
- `reward`
- corresponding `next_*` state variables
- `done`

The dataset is synthetic but deliberately shaped like an operational transition log so that students can practise the data-engineering steps required before modeling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "delivery_rl_transition_log.csv"
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

## Understanding the Code

This cell loads the transition log with Pandas.

- `pd.read_csv(...)` reads the CSV file into a DataFrame.
- `len(df)` checks the number of transition records.
- `len(df.columns)` checks how many fields were loaded.
- `df.head()` displays the first rows so we can inspect the structure.

### Observation

Before modeling, confirm that the file loaded successfully and that the columns represent the current state, action, reward, next state, and episode information we expect.


## First Rule of ML Engineering: Inspect Before Modeling

Do not immediately train a model.

First ask:

- What columns exist?
- What data types do they have?
- Are values missing?
- Are there duplicates?
- Are values in sensible ranges?
- What actions appear?
- How many episodes exist?
- Does the transition structure make sense?

This habit is just as important in RL as in supervised learning.


In [ ]:
df.info()

## Understanding the Code

`df.info()` gives a structural summary of the dataset.

It helps us inspect:

- column names
- non-null counts
- data types
- memory usage

### Observation

Look for columns whose data type does not match its intended meaning, and check whether important columns contain missing values.


In [ ]:
df.describe(include="all").T

## Understanding the Code

`df.describe(include="all").T` creates summary statistics for the dataset and transposes the result so each variable appears as a row.

For numerical columns, inspect measures such as mean, minimum, and maximum. For categorical columns, inspect counts and common values.

### Observation

Look for unusual ranges or categories that may require investigation before modeling.


In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

## Understanding the Code

This cell checks for missing values.

- `isna()` identifies missing entries.
- `sum()` counts them by column.
- `sort_values(...)` puts the columns with the most missing values first.
- `head(20)` limits the display to the first 20 results.

### Observation

Pay particular attention to missing values in state, action, reward, and next-state columns because they directly affect the learning signal.


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Unique episodes:", df["episode_id"].nunique())
print("Actions:", sorted(df["action"].dropna().unique()))
print("Done counts:")
print(df["done"].value_counts())

## Understanding the Code

`value_counts()` shows how frequently each action appears in the dataset.

- `dropna=False` keeps missing action values visible.
- The result helps us understand which actions are common and which are rare.

### Why this matters

Offline RL can struggle when some actions have very little logged experience.

### Observation

Identify the most common and least common actions and consider what that means for the coverage of the dataset.


## Check Numerical Ranges

A data value can be non-missing but still be wrong.

Examples:

- battery should generally be between 0 and 100 percent,
- scores may be expected between 0 and 1,
- distance should not be negative,
- `done` should represent a terminal indicator.

Range checks are a basic form of data validation.


In [ ]:
checks = {
    "battery_below_0": (df["battery_pct"] < 0).sum(),
    "battery_above_100": (df["battery_pct"] > 100).sum(),
    "demand_outside_0_1": ((df["demand_score"] < 0) | (df["demand_score"] > 1)).sum(),
    "traffic_outside_0_1": ((df["traffic_score"] < 0) | (df["traffic_score"] > 1)).sum(),
    "negative_distance": (df["distance_to_goal_km"] < 0).sum(),
    "negative_pending_orders": (df["pending_orders"] < 0).sum(),
}
checks

## Understanding the Code

This cell performs **range validation** on important numerical features.

Each expression counts values that violate an expected rule, such as battery outside `0–100`, scores outside `0–1`, or negative distances.

### Observation

A zero count means no violations were found for that rule. Non-zero counts require investigation before training.


## Check the Actions

An RL model cannot learn an action that is represented inconsistently.

For example, these might accidentally become separate categories:

`Deliver`, `deliver`, `DELIVER`, ` deliver `

We should standardize categorical text before encoding it.


In [ ]:
df["action"].value_counts(dropna=False)

## Understanding the Code

`value_counts()` shows how frequently each action appears in the dataset.

- `dropna=False` keeps missing action values visible.
- The result helps us understand which actions are common and which are rare.

### Why this matters

Offline RL can struggle when some actions have very little logged experience.

### Observation

Identify the most common and least common actions and consider what that means for the coverage of the dataset.


In [ ]:
df["action"] = df["action"].astype(str).str.strip().str.lower()
df["weather_condition"] = df["weather_condition"].astype(str).str.strip().str.lower()
df["zone"] = df["zone"].astype(str).str.strip()

print(df["action"].value_counts())

## Understanding the Code

`value_counts()` shows how frequently each action appears in the dataset.

- `dropna=False` keeps missing action values visible.
- The result helps us understand which actions are common and which are rare.

### Why this matters

Offline RL can struggle when some actions have very little logged experience.

### Observation

Identify the most common and least common actions and consider what that means for the coverage of the dataset.


## Check Transition Consistency

A transition dataset should have current-state and next-state fields that make sense together.

We also need to understand whether rows are sequentially ordered within episodes.

This matters because RL is about **sequences**, not independent rows.


In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values(["episode_id", "timestamp"]).reset_index(drop=True)

print("Missing timestamps:", df["timestamp"].isna().sum())
print(df[["episode_id", "timestamp", "action", "reward", "done"]].head(10))

## Understanding the Code

This cell checks for missing values.

- `isna()` identifies missing entries.
- `sum()` counts them by column.
- `sort_values(...)` puts the columns with the most missing values first.
- `head(20)` limits the display to the first 20 results.

### Observation

Pay particular attention to missing values in state, action, reward, and next-state columns because they directly affect the learning signal.


## A Very Important RL Data Question

In ordinary supervised learning, randomly shuffling rows is often convenient.

In RL, we must be much more careful.

Rows belong to **episodes and trajectories**. Time order and the relationship between current and next state matter.

Randomly splitting transitions can also create leakage or overly optimistic evaluation if nearly identical parts of the same trajectory appear in both training and validation sets.


## Episode-Level Thinking

We can inspect how long episodes are.


In [ ]:
episode_lengths = df.groupby("episode_id").size()
print(episode_lengths.describe())
print("Number of episodes:", len(episode_lengths))

## Understanding the Code

This cell groups records so we can compare reward patterns across actions or operating conditions.

Grouping helps us describe what appears in the data, but it does not by itself prove that an action caused a particular outcome.

### Observation

Look for meaningful differences in reward patterns and consider whether the groups have enough observations to support the comparison.


## Reward Distribution

Before training, understand what the reward signal looks like.

Ask:

- Are rewards mostly positive or negative?
- Are there extreme values?
- Which actions tend to receive which rewards?

This is not yet proof that one action is better. It is descriptive exploration.


In [ ]:
print(df["reward"].describe())

plt.figure(figsize=(8, 4))
plt.hist(df["reward"], bins=30)
plt.xlabel("Reward")
plt.ylabel("Frequency")
plt.title("Reward Distribution")
plt.show()

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


In [ ]:
df.groupby("action")["reward"].agg(["count", "mean", "median", "min", "max"]).sort_values("mean", ascending=False)

## Understanding the Code

This cell groups records so we can compare reward patterns across actions or operating conditions.

Grouping helps us describe what appears in the data, but it does not by itself prove that an action caused a particular outcome.

### Observation

Look for meaningful differences in reward patterns and consider whether the groups have enough observations to support the comparison.


### Interpretation warning

The table above is **descriptive**, not a causal evaluation of the actions.

If `deliver` has a higher average reward, that does not automatically mean:

> “If we force the robot to deliver more often, total performance will improve.”

The recorded action may have been chosen in different states from the other actions. RL evaluation must account for state, trajectory, and policy effects.


# Part 14 — Preparing the CSV for a Learning Experiment

We now build a clean state representation.

For a beginner, think of this as answering:

> **What information will the model receive before it chooses an action?**

We will start with numerical state features so the workflow remains transparent.


## Selecting State Features

For our teaching experiment we will use:

- `hour_of_day`
- `battery_pct`
- `demand_score`
- `traffic_score`
- `distance_to_goal_km`
- `pending_orders`

These are the **current-state features**.

We deliberately do not use the reward or next-state columns as input features. Doing so would leak information from the outcome into the state representation.


In [ ]:
state_features = [
    "hour_of_day",
    "battery_pct",
    "demand_score",
    "traffic_score",
    "distance_to_goal_km",
    "pending_orders"
]

next_state_features = [
    "next_battery_pct",
    "next_demand_score",
    "next_traffic_score",
    "next_distance_to_goal_km",
    "next_pending_orders"
]

print("State features:", state_features)

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


## Encode the Action

Neural networks work with numbers.

Our action column contains words. We need a consistent mapping.

For example:

```text
deliver  → 0
recharge → 1
reroute  → 2
wait     → 3
```

The exact numbers do not carry meaning. They are identifiers.


In [ ]:
action_names = sorted(df["action"].unique())
action_to_id = {name: i for i, name in enumerate(action_names)}
id_to_action = {i: name for name, i in action_to_id.items()}

df["action_id"] = df["action"].map(action_to_id)

print(action_to_id)
print(df[["action", "action_id"]].drop_duplicates().sort_values("action_id"))

## Understanding the Code

This cell converts action labels into numerical representations that a model can use.

The important rule is consistency: the same action must always map to the same representation.

### Observation

Write down the mapping and make sure you can translate each encoded value back to its original action.


## Scale Numerical Features

Our state features have different units.

For example:

- battery may be around 0–100,
- demand is around 0–1,
- pending orders may be whole numbers.

Neural networks often train more comfortably when numerical inputs are on comparable scales.

We will use `StandardScaler` on the **training data only**.


In [ ]:
from sklearn.preprocessing import StandardScaler

# We will split by episode first so that trajectories do not cross the split.
episodes = df["episode_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(42)
rng.shuffle(episodes)

cut = int(0.8 * len(episodes))
train_episodes = set(episodes[:cut])
val_episodes = set(episodes[cut:])

train_df = df[df["episode_id"].isin(train_episodes)].copy()
val_df = df[df["episode_id"].isin(val_episodes)].copy()

scaler = StandardScaler()
train_df[state_features] = scaler.fit_transform(train_df[state_features])
val_df[state_features] = scaler.transform(val_df[state_features])

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))

## Understanding the Code

This cell scales numerical state features so variables measured in different units can be processed more consistently by the model.

The scaler should be fitted on the training data and then reused to transform validation or test data.

### Observation

Scaling changes the numerical representation, not the meaning of the feature.


## Why Fit the Scaler Only on Training Data?

If we calculate scaling statistics using the validation set too, information from the validation data influences preprocessing.

The general rule is:

> **Fit preprocessing on training data; apply the learned transformation to validation/test data.**

This is one of the most important habits you learned in ordinary Machine Learning, and it still matters in RL data experiments.


# Part 15 — Offline / Fitted Q-Learning With the Transition CSV

## Important Concept Before Code

The CSV is a **recorded transition dataset**. We are not letting our model freely interact with the real world.

That means this practical is different from online DQN training.

We will demonstrate a simplified **offline/fitted Q-learning-style** workflow:

1. use recorded transitions,
2. estimate next-state Q-values,
3. construct Bellman targets,
4. train a neural network to fit those targets,
5. repeat the process.

This is useful for teaching the connection between RL theory and real data.


## Why Not Just Use the CSV Like Classification?

A classification model might learn:

`state → action label`

But RL asks a different question:

`state + action → expected long-term return`

The recorded reward and next state are therefore central to the learning problem.

This distinction is essential.


## Build Arrays for the Experiment

We need:

- current state matrix `X`
- action IDs
- rewards
- next-state matrix
- terminal indicators

The next-state features are kept in their original scale for the moment, then we will apply the same fitted scaler used for the current-state features.


In [ ]:
# Build unscaled copies for clear transformation steps
raw_train = df[df["episode_id"].isin(train_episodes)].copy()
raw_val = df[df["episode_id"].isin(val_episodes)].copy()

# Fit using current-state training features
scaler = StandardScaler()
X_train = scaler.fit_transform(raw_train[state_features])
X_val = scaler.transform(raw_val[state_features])

# Use the same scaler statistics for next-state features where dimensions correspond.
next_scaler = scaler
X_next_train = next_scaler.transform(raw_train[[
    "next_battery_pct", "next_demand_score", "next_traffic_score",
    "next_distance_to_goal_km", "next_pending_orders"
]].rename(columns={
    "next_battery_pct": "battery_pct",
    "next_demand_score": "demand_score",
    "next_traffic_score": "traffic_score",
    "next_distance_to_goal_km": "distance_to_goal_km",
    "next_pending_orders": "pending_orders"
}).assign(hour_of_day=raw_train["hour_of_day"].to_numpy())[state_features])

# The hour feature does not have a next_* column, so this simple classroom
# example keeps the current hour for the next-state representation.
X_next_train[:, 0] = X_train[:, 0]

X_next_val = next_scaler.transform(raw_val[[
    "next_battery_pct", "next_demand_score", "next_traffic_score",
    "next_distance_to_goal_km", "next_pending_orders"
]].rename(columns={
    "next_battery_pct": "battery_pct",
    "next_demand_score": "demand_score",
    "next_traffic_score": "traffic_score",
    "next_distance_to_goal_km": "distance_to_goal_km",
    "next_pending_orders": "pending_orders"
}).assign(hour_of_day=raw_val["hour_of_day"].to_numpy())[state_features])
X_next_val[:, 0] = X_val[:, 0]

a_train = raw_train["action_id"].to_numpy()
r_train = raw_train["reward"].to_numpy(dtype=np.float32)
done_train = raw_train["done"].to_numpy(dtype=np.float32)

a_val = raw_val["action_id"].to_numpy()
r_val = raw_val["reward"].to_numpy(dtype=np.float32)
done_val = raw_val["done"].to_numpy(dtype=np.float32)

print(X_train.shape, X_next_train.shape)

## Understanding the Code

This cell scales numerical state features so variables measured in different units can be processed more consistently by the model.

The scaler should be fitted on the training data and then reused to transform validation or test data.

### Observation

Scaling changes the numerical representation, not the meaning of the feature.


### Why is the next-state construction slightly awkward?

Real datasets are not always perfectly designed for learning.

Our file has `next_*` values for several features but not a `next_hour_of_day` field. For this teaching exercise, we keep the current hour as a simple approximation.

A production dataset should define this transition representation explicitly and consistently.

**Tutor note:** This is an opportunity to teach that data modeling decisions can matter as much as algorithm selection.


## Define a Q-Network for the CSV

The network receives six state features and outputs one Q-value per action.

For example, with four actions:

```text
state (6 numbers)
       ↓
   neural network
       ↓
[Q(deliver), Q(recharge), Q(reroute), Q(wait)]
```


In [ ]:
def build_csv_q_network(state_size, action_size):
    """Build a Keras network for the tabular-style state features."""
    return keras.Sequential([
        layers.Input(shape=(state_size,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(action_size, activation="linear")
    ])

csv_q_net = build_csv_q_network(len(state_features), len(action_names))
csv_q_net.summary()


## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


## Fitted Q-Learning Update

We begin with a simple idea.

For every recorded transition, calculate a target:

`target = reward + gamma × max Q(next_state, next_action)`

unless the transition is terminal.

Then train the network so that the Q-value for the recorded action moves toward that target.


In [ ]:
def fit_q_network(X, actions, rewards, X_next, dones,
                   epochs=15, batch_size=128, gamma=0.95, lr=1e-3):
    model = build_csv_q_network(X.shape[1], len(action_names))
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    loss_fn = keras.losses.MeanSquaredError()

    n = len(X)
    history = []

    for epoch in range(epochs):
        order = np.random.permutation(n)
        epoch_loss = 0.0

        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            states = X[idx].astype(np.float32)
            actions_b = actions[idx].astype(np.int32)
            rewards_b = rewards[idx].astype(np.float32)
            next_states = X_next[idx].astype(np.float32)
            dones_b = dones[idx].astype(np.float32)

            # Bellman target from the current model.
            next_q = model(next_states, training=False).numpy()
            max_next_q = np.max(next_q, axis=1)
            targets = rewards_b + gamma * max_next_q * (1.0 - dones_b)

            with tf.GradientTape() as tape:
                q_values = model(states, training=True)
                action_mask = tf.one_hot(actions_b, depth=len(action_names))
                current_q = tf.reduce_sum(q_values * action_mask, axis=1)
                loss = loss_fn(targets, current_q)

            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            epoch_loss += float(loss.numpy()) * len(idx)

        history.append(epoch_loss / n)

    return model, history

csv_q_net, csv_loss = fit_q_network(X_train, a_train, r_train, X_next_train, done_train)
print("Offline/fitted Q-learning experiment complete.")


## Understanding the Code

This cell applies the Bellman idea to **offline transition data**.

Instead of collecting new experiences from an environment, the learner uses transitions that have already been logged.

The general pattern is `transition → future-value estimate → Bellman target → model update`.

### Observation

Trace one row through the calculation and identify the current state, action, reward, next state, and target.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(csv_loss)
plt.xlabel("Epoch")
plt.ylabel("Bellman fitting loss")
plt.title("Offline Q-Network Training Loss")
plt.show()

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


## What Does the Loss Mean Here?

The loss tells us how far the network's predicted Q-value is from the Bellman target used in this educational experiment.

A decreasing loss means the network is fitting the training targets more closely.

**It does not automatically prove that the learned policy is good in the real world.**

That distinction is extremely important.


## Inspect the Learned Action Preference

We can ask the model which action has the largest predicted Q-value for a sample state.


In [ ]:
sample_state = X_val[:5].astype(np.float32)
sample_q = csv_q_net(sample_state, training=False).numpy()

for i, values in enumerate(sample_q):
    best_id = int(np.argmax(values))
    print(f"Row {i}: best predicted action = {id_to_action[best_id]}")
    print(dict(zip(action_names, np.round(values, 3))))


## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


## Offline RL Warning: Distribution Shift

Suppose the dataset contains mostly states where the recorded operator chose `deliver`.

A learned model may be asked to evaluate an action in a state-action combination that barely or never appeared in the dataset.

That is dangerous because the model may be extrapolating beyond its evidence.

This is one reason **offline RL is a specialized research and engineering area**, not simply “normal supervised learning with a reward column.”


# Part 16 — Reward Engineering

## Why Reward Design Matters

Suppose a company rewards a delivery robot only for the number of completed deliveries.

The system may have an incentive to optimize that metric while ignoring:

- safety,
- battery health,
- customer satisfaction,
- lateness,
- traffic,
- operational constraints.

This is called a **reward specification problem**, and undesirable behaviour caused by optimizing the wrong objective is often discussed under the term **reward hacking**.


## Example

Imagine:

`reward = +10 for every delivery`

A robot might prefer many easy deliveries and ignore difficult but important orders.

A more complete objective could include multiple terms such as:

`delivery_success − lateness_cost − energy_cost − safety_penalty`

But adding more terms does not automatically make the reward correct. The weights and definitions require domain knowledge and validation.


## Exercise — Design a Reward

For a warehouse robot, propose a reward containing:

- one positive component,
- two negative components,
- one safety consideration.

Then answer:

> How could your reward accidentally encourage undesirable behaviour?


In [ ]:
# Student design area
reward_design = {
    "positive": "",
    "negative_1": "",
    "negative_2": "",
    "safety": ""
}

reward_design

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


# Part 17 — Common Beginner Mistakes

### Mistake 1: “RL is just classification.”

No. Classification predicts a label. RL learns sequential decisions where actions affect future states and rewards.

### Mistake 2: “The highest immediate reward is always best.”

No. Long-term return matters.

### Mistake 3: “More exploration is always better.”

No. Exploration is useful for learning, but uncontrolled exploration can be unsafe or inefficient.

### Mistake 4: “A lower training loss proves the policy is good.”

No. We need behaviour-level evaluation and appropriate validation.

### Mistake 5: “A Q-value is a guaranteed score.”

No. It is an estimate produced by the learning process.


## RL Debugging Checklist

When an RL agent behaves strangely, check:

### Environment
- Are actions implemented correctly?
- Are terminal states reachable?
- Are rewards generated correctly?
- Does `reset()` really reset the environment?

### Data
- Are transitions ordered correctly?
- Are current and next states aligned?
- Are actions encoded consistently?
- Are there impossible values?

### Learning
- Is epsilon changing as expected?
- Is gamma reasonable for the task?
- Is the learning rate too large or too small?
- Are targets detached when necessary?
- Is the target network being updated?

### Evaluation
- Are we evaluating without unnecessary exploration?
- Are we using held-out episodes?
- Are we measuring behaviour, not only loss?


# Part 18 — Where Modern RL Fits

The classical ideas in this notebook lead into a much larger family of RL methods.

### Value-based methods

- Q-Learning
- DQN
- Double DQN
- Dueling DQN

### Policy-based methods

- Policy Gradient
- REINFORCE

### Actor-Critic methods

- A2C/A3C-style methods
- PPO
- SAC
- TD3

### Model-based RL

The agent also learns or uses a model of how the environment changes.

### Offline RL

The agent learns from previously collected data rather than freely collecting new experience.

The central concepts you learned — state, action, reward, return, value, policy, exploration, and sequential interaction — remain important across these families.


## PPO at a Conceptual Level

**PPO (Proximal Policy Optimization)** is a popular policy-optimization method.

A beginner does not need to implement PPO immediately.

First understand the idea:

- instead of directly storing Q-values in a table,
- we can learn a policy that outputs action probabilities,
- and improve that policy using experience,
- while constraining updates so that the new policy does not change too aggressively in one step.

This is a useful bridge from Q-learning to modern policy-optimization methods.


# Part 19 — Real-World RL Engineering

A classroom agent can explore freely in a toy grid.

A real robot, financial system, industrial controller, or healthcare system cannot simply try arbitrary actions without consequences.

Real RL projects therefore require attention to:

- safety constraints,
- simulation and digital twins,
- offline evaluation,
- monitoring,
- human override,
- distribution shift,
- reward validation,
- uncertainty,
- reproducibility,
- governance and auditing.


## Simulation Before Reality

A common engineering strategy is:

**real system → build simulator → train/test in simulation → validate carefully → controlled deployment**

Simulation is not automatically identical to reality. The difference between simulated and real behaviour is often discussed as the **sim-to-real gap**.


## RL Is Not Always the Right Tool

A strong AI engineer does not choose RL simply because the problem contains the word “decision.”

Ask first:

- Is there a sequential decision problem?
- Do actions affect future states?
- Can we define meaningful feedback?
- Can we safely collect or simulate experience?
- Is a simpler optimization or supervised approach sufficient?

The algorithm should follow the problem, not the other way around.


# Part 20 — Guided Practical: Build Your Own RL Problem

Choose one scenario:

1. traffic-light control,
2. warehouse robot,
3. game agent,
4. delivery routing,
5. energy management,
6. inventory management.

Fill in the template.


In [ ]:
rl_problem = {
    "problem": "",
    "agent": "",
    "environment": "",
    "state": [],
    "actions": [],
    "reward": "",
    "episode": "",
    "goal": "",
    "safety_constraint": ""
}

rl_problem

## Understanding the Code

This cell implements the practical step introduced in the section above.

- Read the variables and function calls from top to bottom.
- Identify which values represent the current state, action, reward, or learning parameter.
- Run the cell and compare the output with the concept being taught.

### Observation

Do not only check whether the code runs. Explain what the output means in the context of reinforcement learning.


## Practical Extension

Now answer these questions:

1. What information is missing from your state?
2. Could your reward encourage the wrong behaviour?
3. Which actions could be unsafe?
4. What would cause an episode to end?
5. What would exploration look like?
6. How would you evaluate the learned policy?


# Part 21 — Mini Assessment

Answer these without looking back.

### Question 1
What is the difference between an agent and an environment?

### Question 2
What is a state?

### Question 3
Why does an RL system need a reward?

### Question 4
What is the difference between exploration and exploitation?

### Question 5
What does `Q(s,a)` represent?

### Question 6
What is the main idea behind the Bellman equation?

### Question 7
Why can a Q-table become impractical?

### Question 8
What does DQN replace the Q-table with?

### Question 9
Why is the next state important in an RL transition dataset?

### Question 10
Why should training and validation episodes be separated carefully?


## Assessment — Debug This Reasoning

A student says:

> “The `deliver` action has the highest average reward in the CSV, so I will tell the robot to always choose `deliver`.”

What is wrong with this reasoning?

**Tutor answer guide:**

The average reward is descriptive and may reflect the states in which `deliver` was chosen. It does not establish that always choosing `deliver` will maximize long-term return. RL decisions are state-dependent and sequential.


# Final Project — Beginner to Professional

## Project Brief

Design and implement a small reinforcement-learning problem.

### Required sections

1. Problem definition
2. Real-world analogy
3. Agent
4. Environment
5. State representation
6. Action space
7. Reward design
8. Episode definition
9. Baseline strategy
10. Q-Learning or DQN experiment
11. Evaluation metrics
12. Training curve
13. Behaviour analysis
14. Failure cases
15. Reward-hacking risk
16. Safety considerations
17. Limitations
18. Future improvements

### Minimum technical requirement

Your project should contain a working environment and a learning loop. A small GridWorld-style environment is acceptable for beginners; students ready for a larger challenge can use a custom simulation or transition dataset.


# Tutor Guide — Suggested Teaching Sequence

## Session 1: Story Before Mathematics

Spend most of the time on:

- agent,
- environment,
- state,
- action,
- reward,
- episode,
- policy.

Have students classify examples before writing code.

## Session 2: Returns and Q-values

Use small numerical examples. Ask students to calculate returns by hand before running Python.

## Session 3: GridWorld

Build the environment slowly. Let students inspect states, actions and rewards.

## Session 4: Q-Learning

Write the update function line by line and connect every variable to the story.

## Session 5: DQN

Explain why the table stops scaling. Introduce neural networks, replay, and target networks one at a time.

## Session 6: Real-World Data

Return to the transition structure and teach inspection, cleaning, leakage prevention, episode-based splitting, and offline-RL limitations.

## Session 7: Project

Students design their own RL problem and justify every component.


# Final Mental Model

If you remember only one framework from this notebook, remember this:

```text
             CURRENT SITUATION
                    │
                    ▼
                 STATE s
                    │
                    ▼
              ┌──────────┐
              │   AGENT  │
              └────┬─────┘
                   │ chooses
                   ▼
                ACTION a
                   │
                   ▼
             ┌─────────────┐
             │ ENVIRONMENT │
             └──────┬──────┘
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
       REWARD r           NEXT STATE s'
          │                   │
          └─────────┬─────────┘
                    ▼
                 LEARN
                    │
                    └──────► repeat
```

The central RL question is:

> **“Given what I know about the current situation, which action should I take to achieve good long-term outcomes?”**

Everything else in this notebook — returns, value functions, Q-values, Bellman targets, Q-Learning, DQN, replay buffers, policies, exploration, and offline RL — builds on that idea.


# Reflection

Complete these sentences:

1. Before this lesson, I thought reinforcement learning was...
2. Now I understand that reinforcement learning is...
3. The easiest concept for me was...
4. The most difficult concept was...
5. I can now explain `Q(s,a)` as...
6. One real-world RL problem I would like to solve is...
7. One risk I would consider before deploying it is...

**Tutor note:** This reflection is intentionally conceptual. Students should be able to explain RL in plain language before being expected to implement sophisticated algorithms.
